In [1]:
import requests
import pandas as pd
import time


OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5:0.5b"


def check_ollama_server():
    """
    Проверяет доступность локального Ollama-сервера.

    Возвращает:
        bool: True, если сервер доступен, иначе False.
    """
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        return response.status_code == 200
    except requests.exceptions.RequestException:
        return False


def send_prompt_to_ollama(prompt, model=MODEL_NAME):
    """
    Отправляет текстовый запрос в LLM-модель через Ollama HTTP API.

    Аргументы:
        prompt (str): Текст запроса к модели.
        model (str): Название модели Ollama.

    Возвращает:
        str: Ответ LLM-модели.
    """
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }

    try:
        response = requests.post(
            OLLAMA_URL,
            json=payload,
            timeout=120
        )

        response.raise_for_status()
        data = response.json()

        return data.get("response", "").strip()

    except requests.exceptions.RequestException as error:
        return f"Ошибка при обращении к Ollama: {error}"


def run_inference(prompts):
    """
    Выполняет инференс для списка запросов и сохраняет результаты в таблицу.

    Аргументы:
        prompts (list[str]): Список запросов к LLM.

    Возвращает:
        pandas.DataFrame: Таблица с запросами и ответами модели.
    """
    results = []

    for index, prompt in enumerate(prompts, start=1):
        print(f"Выполняется запрос {index}/{len(prompts)}...")

        answer = send_prompt_to_ollama(prompt)

        results.append({
            "Запрос к LLM": prompt,
            "Вывод LLM": answer
        })

        time.sleep(1)

    return pd.DataFrame(results)


prompts = [
    "Кратко объясни, что такое искусственный интеллект.",
    "Назови три основные области применения нейронных сетей.",
    "Объясни простыми словами, что такое машинное обучение.",
    "В чем разница между supervised learning и unsupervised learning?",
    "Напиши короткое определение большой языковой модели.",
    "Какие преимущества есть у локального запуска LLM через Ollama?",
    "Какие недостатки могут быть у маленьких языковых моделей?",
    "Объясни, что такое инференс модели.",
    "Приведи пример использования LLM в бизнесе.",
    "Сделай краткий вывод о пользе локальных LLM-моделей."
]


if not check_ollama_server():
    print("Ollama-сервер недоступен.")
    print("Проверь, что в терминале запущена команда: ollama serve")
else:
    print("Ollama-сервер доступен.")
    inference_report = run_inference(prompts)
    display(inference_report)

Ollama-сервер доступен.
Выполняется запрос 1/10...
Выполняется запрос 2/10...
Выполняется запрос 3/10...
Выполняется запрос 4/10...
Выполняется запрос 5/10...
Выполняется запрос 6/10...
Выполняется запрос 7/10...
Выполняется запрос 8/10...
Выполняется запрос 9/10...
Выполняется запрос 10/10...


,Запрос к LLM,Вывод LLM
0,"Кратко объясни, что такое искусственный интелл...","I'm sorry for any confusion earlier, but as an..."
1,Назови три основные области применения нейронн...,Нейронные сети являются востребованными интелл...
2,"Объясни простыми словами, что такое машинное о...",Вот общая информация о машинном обучении:\n\n1...
3,В чем разница между supervised learning и unsu...,Задачи и методы обучения supervised (обучающая...
4,Напиши короткое определение большой языковой м...,"Возможно, для вас будет понятно, что ""большой""..."
5,Какие преимущества есть у локального запуска L...,LLM (Lattice Learning Model) через Ollama (оно...
6,Какие недостатки могут быть у маленьких языков...,Маленькие языковые модальные модели часто обла...
7,"Объясни, что такое инференс модели.",Инференс — это технология искусственного интел...
8,Приведи пример использования LLM в бизнесе.,LLM (Large Language Model) - это искусственный...
9,Сделай краткий вывод о пользе локальных LLM-мо...,"Локальные LLM-модели, такие как T5 и Anthropic..."


In [3]:
inference_report.to_csv("ollama_inference_report.csv", index=False, encoding="utf-8-sig")
inference_report.to_markdown("ollama_inference_report.md", index=False)

print("Отчет сохранен в файлы:")
print("1. ollama_inference_report.csv")
print("2. ollama_inference_report.md")

Отчет сохранен в файлы:
1. ollama_inference_report.csv
2. ollama_inference_report.md
